# ML-09 — Week 6 Validation Audit

## Lane 2: Refresh / Content Opportunity Scoring

This notebook audits the Week-5 model before it is used for later decision-support work.

The audit has four goals:

1. Review two real findings from the FlyRank March 2026 research paper and ask constructive methodology questions.
2. Re-run the Week-5 model under a genuinely time-aware validation design.
3. Audit every model feature for label, future-window, decision-derived, identifier, and date leakage.
4. Inspect real false positives and false negatives and rewrite claims so they match the evidence.

**Important interpretation boundary:** the model ranks content against a future-performance proxy. The proxy is not a ground-truth `needs_refresh` label and does not establish that refreshing a page causes recovery.

This notebook deliberately uses careful terms such as **observed**, **measured**, **associated with**, **directional**, and **decision-support**.


## 0. Dependency map and Week-5 artifacts reused

**W1 →** research question and Lane 2 framing  
**W2 →** ranking/scoring task, Precision@K, human-review framing  
**W3 →** data contract and leakage exclusions  
**W4 →** transparent baseline rule and forward-performance proxy  
**W5 →** Logistic Regression + Random Forest using the approved five warehouse signals and documented transforms  
**W6 →** validation audit, honest time-aware test, leakage audit, failure analysis, claim audit  
**W7 →** action playbook using the validated ranking evidence  
**W8 →** public-safe capstone research paper

### Exact Week-5 artifacts reused

- Decision snapshot grain: one row per `client_hash_id × content_hash_id` at a month-end decision date.
- Week-4 baseline rule: `2 × I(impressions >= 500) + I(clicks >= 10)`.
- Week-4/5 future outcome: next month's average daily GSC impressions is below the decision-day GSC impressions.
- Week-5 model family: Logistic Regression and Random Forest.
- Week-5 approved predictive signals: GSC impressions, GSC clicks, GSC average position, GA4 sessions, and GA4 users, with the same documented transformations.
- Primary ranking metric: Precision@10, Precision@20, Precision@50, with the base rate printed beside them.

**Validation improvement:** Week 5 used a grouped client holdout inside the March decision snapshot. This audit keeps that result as the **before** design and adds a true chronological train/test design: February decision → March outcome for training, then March decision → April outcome for testing.


In [3]:
# Setup
# This notebook is intended to run in Colab or another environment with access
# to the gated FlyRank warehouse. No token is hard-coded.

!pip -q install duckdb

import os
import duckdb
import numpy as np
import pandas as pd

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Colab: store the token in Colab Secrets under the name "flyrank".
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("flyrank")
except Exception:
    hf_token = os.environ.get("FLYRANK_HF_TOKEN")

if not hf_token:
    raise RuntimeError(
        "FlyRank warehouse access is required for the real Week-6 validation. "
        "In Colab, add the Hugging Face token to Colab Secrets as 'flyrank', "
        "then run all cells again. Do not paste credentials into the notebook."
    )

con = duckdb.connect()
con.execute(f"""
CREATE OR REPLACE SECRET flyrank_hf (
    TYPE huggingface,
    TOKEN '{hf_token}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FEB_PATH = f"{REL}/fact_content_daily_performance/month=2026-02/*.parquet"
MAR_PATH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"
APR_PATH = f"{REL}/fact_content_daily_performance/month=2026-04/*.parquet"

print("Connected to FlyRank warehouse.")
print("Training decision snapshot:", "2026-02-28")
print("Training outcome month:", "2026-03")
print("Test decision snapshot:", "2026-03-31")
print("Test outcome month:", "2026-04")

Connected to FlyRank warehouse.
Training decision snapshot: 2026-02-28
Training outcome month: 2026-03
Test decision snapshot: 2026-03-31
Test outcome month: 2026-04


## 1. Two real FlyRank paper findings + constructive methodology questions

The questions below are not intended as criticism of the authors. They identify where additional validation or clearer provenance would make an observational finding easier to interpret.


### Finding 1 — The Anatomy of Growing Content

The paper reports that growing pages in its portfolio were **37.6% longer** on average (3.2K vs 2.3K words) and **20% younger** (184 vs 230 days) than declining pages. It reports 74,187 rising pages versus 45,272 falling pages and explicitly describes the comparison as observational. citeturn0file1L124-L144

**Methodology question:** Could the observed word-count difference partly reflect content age, content type, topic mix, or selection into the growing/declining cohorts? A useful strengthening step would be to show whether the relationship remains similar within comparable age/content/topic strata rather than interpreting the cross-sectional difference as evidence that adding length itself improves performance.

**Why this is constructive:** the paper already labels the result observational. The question is about separating correlated characteristics and checking whether the measured association is stable across relevant subgroups.


### Finding 2 — The Age-Freshness Matrix

The paper reports that the `365+ × 0-30 days fresh` group has a health score of **44.62**, close to **44.12** for the `31-90 days age × 0-30 days fresh` group. It describes the older refreshed group as performing nearly as well as the young-fresh group, while warning that the small `365+ × 361+` survivor cell should not be treated as a headline decay proof point. citeturn0file1L356-L375

**Methodology question:** Because refreshes are not randomly assigned, how much of the difference between refreshed and long-unchanged older content could be explained by selection into refreshes, prior performance, topic mix, or editorial importance? A stronger design would report sample sizes for each cell and, where possible, compare refreshed pages with matched or otherwise comparable unrefreshed pages over a defined pre/post window.

**Why this is constructive:** the paper already flags survivor bias in the small tail. The question extends that same caution to selection bias around which pages were refreshed.


## 2. Load the actual warehouse snapshots and future outcomes

### Honest time boundary

For the **training period**, only information available on or before **2026-02-28** is used as features. The label is derived from March 2026.

For the **test period**, only information available on or before **2026-03-31** is used as features. The label is derived from April 2026.

Thus the model never receives March/April outcome information as an input when making the corresponding decision-time prediction.


In [4]:
def load_snapshot(path, decision_date):
    return con.sql(f"""
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            gsc_data_available,
            ga4_data_available,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position,
            ga4_sessions,
            ga4_users
        FROM read_parquet('{path}')
        WHERE report_date = DATE '{decision_date}'
    """).df()


def load_forward_outcome(path):
    return con.sql(f"""
        SELECT
            client_hash_id,
            content_hash_id,
            AVG(gsc_impressions) AS future_avg_daily_impressions,
            COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS future_days_observed
        FROM read_parquet('{path}')
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    """).df()


feb_snapshot = load_snapshot(FEB_PATH, "2026-02-28")
mar_snapshot = load_snapshot(MAR_PATH, "2026-03-31")

mar_outcome = load_forward_outcome(MAR_PATH)
apr_outcome = load_forward_outcome(APR_PATH)

print("February decision snapshot:", feb_snapshot.shape)
print("March decision snapshot:", mar_snapshot.shape)
print("March forward outcome:", mar_outcome.shape)
print("April forward outcome:", apr_outcome.shape)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

February decision snapshot: (274038, 10)
March decision snapshot: (331436, 10)
March forward outcome: (176738, 4)
April forward outcome: (194760, 4)


In [6]:
from google.colab import userdata

# Attempt to retrieve the 'flyrank' secret
retrieved_hf_token = userdata.get("flyrank")

if retrieved_hf_token:
    print(f"Hugging Face token successfully retrieved from Colab secrets. First 5 characters: {retrieved_hf_token[:5]}...")
else:
    print("Hugging Face token 'flyrank' not found in Colab secrets.")

# If you want to see the full token (be cautious about sharing):
# print(f"Full token: {retrieved_hf_token}")

Hugging Face token successfully retrieved from Colab secrets. First 5 characters: hf_dp...


In [7]:
def attach_future_decline(snapshot, future_outcome):
    frame = snapshot.merge(
        future_outcome,
        on=["client_hash_id", "content_hash_id"],
        how="inner",
    ).copy()

    frame["future_decline_proxy"] = (
        frame["future_avg_daily_impressions"] < frame["gsc_impressions"]
    ).astype(int)

    return frame


train_raw = attach_future_decline(feb_snapshot, mar_outcome)
test_raw = attach_future_decline(mar_snapshot, apr_outcome)

print(f"Training rows with March outcome: {len(train_raw):,}")
print(f"Test rows with April outcome: {len(test_raw):,}")

print(
    f"Training rows without March outcome: "
    f"{len(feb_snapshot) - len(train_raw):,}"
)
print(
    f"Test rows without April outcome: "
    f"{len(mar_snapshot) - len(test_raw):,}"
)

print(
    f"Training base rate: {train_raw['future_decline_proxy'].mean():.3f}"
)
print(
    f"Test base rate: {test_raw['future_decline_proxy'].mean():.3f}"
)


Training rows with March outcome: 150,802
Test rows with April outcome: 176,441
Training rows without March outcome: 123,236
Test rows without April outcome: 154,995
Training base rate: 0.221
Test base rate: 0.371


## 3. Week-5 feature construction, with train-only preprocessing

The Week-5 feature set is reused rather than expanded.

The five approved raw signals are:

- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `ga4_sessions`
- `ga4_users`

Week 5 then used documented transformations for missing position/GA4 availability and the `gsc_ctr` ratio. The important validation correction here is that **imputation parameters are fitted on the training period only**. The test-period median is never allowed to influence training preprocessing.


In [8]:
FEATURE_COLS = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_ctr",
    "gsc_avg_position_clean",
    "has_position_data",
    "ga4_sessions_clean",
    "ga4_users_clean",
    "has_ga4",
]


def prepare_features(frame, position_median=None):
    x = frame.copy()

    x["gsc_avg_position_clean"] = x["gsc_avg_position"].replace(0, np.nan)
    x["has_position_data"] = (
        x["gsc_avg_position"].notna() &
        x["gsc_avg_position"].ne(0)
    ).astype(int)

    if position_median is None:
        position_median = x["gsc_avg_position_clean"].median()

    x["gsc_avg_position_clean"] = (
        x["gsc_avg_position_clean"].fillna(position_median)
    )

    x["has_ga4"] = (
        x["ga4_data_available"]
        .fillna(False)
        .astype(bool)
        .astype(int)
    )

    x["ga4_sessions_clean"] = (
        x["ga4_sessions"]
        .where(x["ga4_data_available"] == True, 0)
        .fillna(0)
    )

    x["ga4_users_clean"] = (
        x["ga4_users"]
        .where(x["ga4_data_available"] == True, 0)
        .fillna(0)
    )

    x["gsc_ctr"] = np.where(
        x["gsc_impressions"] > 0,
        x["gsc_clicks"] / x["gsc_impressions"],
        0.0,
    )

    return x, position_median


train_df, train_position_median = prepare_features(train_raw)
test_df, _ = prepare_features(
    test_raw,
    position_median=train_position_median,
)

print("Features used:")
print(FEATURE_COLS)
print("Training position median:", train_position_median)


Features used:
['gsc_impressions', 'gsc_clicks', 'gsc_ctr', 'gsc_avg_position_clean', 'has_position_data', 'ga4_sessions_clean', 'ga4_users_clean', 'has_ga4']
Training position median: 6.4520547945205475


## 4. Week-5 validation design — BEFORE

Week 5 used a **70/30 `GroupShuffleSplit` by client** on the March 31 decision snapshot. That prevents a client from appearing in both the model-training and held-out sets.

This section reproduces that design on the March decision snapshot so the audit has a documented **before** condition. The same baseline and Week-5 model definitions are used.

The grouped design is useful because it tests transfer to clients not seen during training, but it is not a chronological deployment simulation. The new design in the next section addresses that specific limitation.


In [9]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores), kind="stable")
    top_k = order[:min(k, len(order))]
    return float(np.asarray(y_true)[top_k].mean())


K_VALUES = [10, 20, 50]


def baseline_score(df):
    visible = (df["gsc_impressions"] >= 500).astype(int)
    active = (df["gsc_clicks"] >= 10).astype(int)
    return visible * 2 + active


before_base = attach_future_decline(mar_snapshot, apr_outcome)
before, before_median = prepare_features(before_base)

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=RANDOM_SEED,
)
before_train_idx, before_test_idx = next(
    gss.split(before, groups=before["client_hash_id"])
)

before_train = before.iloc[before_train_idx].reset_index(drop=True)
before_test = before.iloc[before_test_idx].reset_index(drop=True)

assert not (
    set(before_train["client_hash_id"]) &
    set(before_test["client_hash_id"])
)

X_before_train = before_train[FEATURE_COLS]
y_before_train = before_train["future_decline_proxy"]
X_before_test = before_test[FEATURE_COLS]
y_before_test = before_test["future_decline_proxy"]

before_logreg = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_SEED
    )),
])
before_logreg.fit(X_before_train, y_before_train)
before_logreg_scores = before_logreg.predict_proba(X_before_test)[:, 1]

before_rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=25,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
before_rf.fit(X_before_train, y_before_train)
before_rf_scores = before_rf.predict_proba(X_before_test)[:, 1]

before_results = []
for name, scores in [
    ("Week-4 baseline", baseline_score(before_test)),
    ("Week-5 Logistic Regression", before_logreg_scores),
    ("Week-5 Random Forest", before_rf_scores),
]:
    row = {"Method": name}
    for k in K_VALUES:
        row[f"Precision@{k}"] = precision_at_k(
            y_before_test, scores, k
        )
    row["Base Rate"] = float(y_before_test.mean())
    before_results.append(row)

before_table = pd.DataFrame(before_results).set_index("Method")
before_table.round(3)


,Precision@10,Precision@20,Precision@50,Base Rate
Method,,,,
Week-4 baseline,0.8,0.75,0.72,0.405
Week-5 Logistic Regression,0.8,0.65,0.76,0.405
Week-5 Random Forest,0.8,0.75,0.68,0.405


## 5. Honest time-aware validation — AFTER

The chronological test is:

**February 28 decision-time features → March outcome**  
**March 31 decision-time features → April outcome**

The model is trained only on the earlier decision period and evaluated on the later decision period. No March/April future outcome is used to create test features.

This is closer to the intended deployment question: *if the system learned from an earlier period, how well does its ranking carry into the next decision period?*


In [10]:
X_train = train_df[FEATURE_COLS]
y_train = train_df["future_decline_proxy"]

X_test = test_df[FEATURE_COLS]
y_test = test_df["future_decline_proxy"]

time_logreg = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_SEED
    )),
])
time_logreg.fit(X_train, y_train)
time_logreg_scores = time_logreg.predict_proba(X_test)[:, 1]

time_rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=25,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
time_rf.fit(X_train, y_train)
time_rf_scores = time_rf.predict_proba(X_test)[:, 1]

time_results = []
for name, scores in [
    ("Week-4 baseline", baseline_score(test_df)),
    ("Week-5 Logistic Regression", time_logreg_scores),
    ("Week-5 Random Forest", time_rf_scores),
]:
    row = {"Method": name}
    for k in K_VALUES:
        row[f"Precision@{k}"] = precision_at_k(
            y_test, scores, k
        )
    row["Base Rate"] = float(y_test.mean())
    time_results.append(row)

time_table = pd.DataFrame(time_results).set_index("Method")

print("HONEST TIME-AWARE RESULTS")
display(time_table.round(3))


HONEST TIME-AWARE RESULTS


,Precision@10,Precision@20,Precision@50,Base Rate
Method,,,,
Week-4 baseline,0.8,0.75,0.70,0.371
Week-5 Logistic Regression,1.0,0.85,0.84,0.371
Week-5 Random Forest,0.8,0.70,0.78,0.371


In [11]:
# BEFORE / AFTER comparison.
# The rows intentionally retain the validation design and evaluation population
# in the labels so the reader does not mistake the two numbers for identical
# experiments.

comparison = pd.concat([
    before_table.loc[["Week-4 baseline", "Week-5 Logistic Regression", "Week-5 Random Forest"]]
        .assign(Validation="BEFORE: March grouped client holdout"),
    time_table.loc[["Week-4 baseline", "Week-5 Logistic Regression", "Week-5 Random Forest"]]
        .assign(Validation="AFTER: Feb→Mar train, Mar→Apr test"),
]).reset_index()

comparison = comparison[
    ["Validation", "Method"] +
    [f"Precision@{k}" for k in K_VALUES] +
    ["Base Rate"]
]

display(comparison.round(3))


,Validation,Method,Precision@10,Precision@20,Precision@50,Base Rate
0,BEFORE: March grouped client holdout,Week-4 baseline,0.8,0.75,0.72,0.405
1,BEFORE: March grouped client holdout,Week-5 Logistic Regression,0.8,0.65,0.76,0.405
2,BEFORE: March grouped client holdout,Week-5 Random Forest,0.8,0.75,0.68,0.405
3,"AFTER: Feb→Mar train, Mar→Apr test",Week-4 baseline,0.8,0.75,0.70,0.371
4,"AFTER: Feb→Mar train, Mar→Apr test",Week-5 Logistic Regression,1.0,0.85,0.84,0.371
5,"AFTER: Feb→Mar train, Mar→Apr test",Week-5 Random Forest,0.8,0.70,0.78,0.371


### Interpretation rule

The time-aware result is the primary validation result for this audit.

If Precision@K decreases relative to the grouped Week-5 result, that decrease is **not hidden or repaired by changing thresholds after seeing the result**. It is evidence that chronological generalization is harder than the earlier grouped design suggested.

The baseline is included in both designs so model performance is interpreted relative to the same transparent rule.


## 6. Feature-level leakage audit

The audit follows the repository's leakage taxonomy:

1. **Label-derived leakage** — no `trend_direction`, `trend_pct`, or target column is a feature.
2. **Future-window leakage** — March/April outcome columns are used only to create evaluation labels after decision-time features are constructed.
3. **Decision-derived leakage** — no existing product score, flag, or model output is used as an input.
4. **Identifier leakage** — `client_hash_id` and `content_hash_id` are grouping/join keys only.
5. **Preprocessing leakage** — the position imputation median for the chronological model is fitted on February training data and then reused for March test data.
6. **Date leakage** — `report_date` is not a model feature.

The code below makes these checks executable rather than leaving them as prose.


In [12]:
# Explicit forbidden columns.
FORBIDDEN_FEATURES = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "future_decline_proxy",
    "future_avg_daily_impressions",
    "future_days_observed",
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "april_avg_daily_impressions",
}

assert not (set(FEATURE_COLS) & FORBIDDEN_FEATURES)

# Feature provenance table.
provenance = pd.DataFrame([
    ["gsc_impressions", "GSC snapshot at decision date", "Allowed", "Available at decision time"],
    ["gsc_clicks", "GSC snapshot at decision date", "Allowed", "Available at decision time"],
    ["gsc_ctr", "gsc_clicks / gsc_impressions", "Allowed", "Derived only from decision-time inputs"],
    ["gsc_avg_position_clean", "Decision-date GSC position; zero treated as missing", "Allowed", "Imputation median fitted on training period only"],
    ["has_position_data", "Whether position was observed", "Allowed", "Decision-time availability flag"],
    ["ga4_sessions_clean", "GA4 sessions respecting availability", "Allowed", "Decision-time source field"],
    ["ga4_users_clean", "GA4 users respecting availability", "Allowed", "Decision-time source field"],
    ["has_ga4", "GA4 availability flag", "Allowed", "Decision-time source field"],
], columns=["Feature", "Origin", "Leakage verdict", "Reason"])

display(provenance)

print("Forbidden-feature intersection:", set(FEATURE_COLS) & FORBIDDEN_FEATURES)
assert len(set(FEATURE_COLS) & FORBIDDEN_FEATURES) == 0


,Feature,Origin,Leakage verdict,Reason
0,gsc_impressions,GSC snapshot at decision date,Allowed,Available at decision time
1,gsc_clicks,GSC snapshot at decision date,Allowed,Available at decision time
2,gsc_ctr,gsc_clicks / gsc_impressions,Allowed,Derived only from decision-time inputs
3,gsc_avg_position_clean,Decision-date GSC position; zero treated as mi...,Allowed,Imputation median fitted on training period only
4,has_position_data,Whether position was observed,Allowed,Decision-time availability flag
5,ga4_sessions_clean,GA4 sessions respecting availability,Allowed,Decision-time source field
6,ga4_users_clean,GA4 users respecting availability,Allowed,Decision-time source field
7,has_ga4,GA4 availability flag,Allowed,Decision-time source field


Forbidden-feature intersection: set()


In [13]:
# Deliberate leakage sanity check.
# A future-derived feature should be dramatically predictive if accidentally
# introduced, which makes this a useful test of the audit harness.

leak_test = test_df.copy()
leak_test["deliberately_leaky_feature"] = leak_test["future_decline_proxy"]

leak_accuracy = (
    leak_test["deliberately_leaky_feature"]
    == leak_test["future_decline_proxy"]
).mean()

print(f"Deliberate leakage sanity-check accuracy: {leak_accuracy:.3f}")
assert leak_accuracy == 1.0

# The real model feature list remains clean.
assert "deliberately_leaky_feature" not in FEATURE_COLS
print("Real feature set remains leakage-safe after the demonstration.")


Deliberate leakage sanity-check accuracy: 1.000
Real feature set remains leakage-safe after the demonstration.


## 7. Real failure analysis from the honest validation

The following tables use actual predictions from the **time-aware test period**.

For public-safe reporting, identifiers are immediately replaced by `Example A`, `Example B`, etc. No client names, domains, URLs, queries, or raw content are displayed.

A false positive means the item was ranked in the review queue but did **not** match the future-decline proxy. A false negative here means a proxy-positive item received a comparatively low model score. Neither category proves that the model made an editorially wrong decision because the proxy is not a ground-truth refresh label.


In [14]:
# Choose the stronger Week-5 model by the pre-declared Precision@20 rule.
lift_p20 = (
    time_table.loc["Week-5 Random Forest", "Precision@20"]
    - time_table.loc["Week-5 Logistic Regression", "Precision@20"]
)

MEANINGFUL_LIFT = 0.02

if lift_p20 >= MEANINGFUL_LIFT:
    PRIMARY_MODEL_NAME = "Week-5 Random Forest"
    primary_model = time_rf
    primary_scores = time_rf_scores
else:
    PRIMARY_MODEL_NAME = "Week-5 Logistic Regression"
    primary_model = time_logreg
    primary_scores = time_logreg_scores

print(
    f"Primary model under the honest split: {PRIMARY_MODEL_NAME}. "
    f"Random Forest Precision@20 lift over Logistic Regression = {lift_p20:.3f}."
)

eval_df = test_df.copy()
eval_df["model_score"] = primary_scores
eval_df["model_rank"] = (
    eval_df["model_score"]
    .rank(ascending=False, method="first")
    .astype(int)
)

top20 = eval_df.sort_values("model_score", ascending=False).head(20)
false_positives = top20[top20["future_decline_proxy"] == 0].copy()

false_negatives = (
    eval_df[eval_df["future_decline_proxy"] == 1]
    .sort_values("model_score", ascending=True)
    .head(5)
    .copy()
)

print(f"False positives in top 20: {len(false_positives)}")
print(f"Low-ranked proxy positives inspected as false-negative examples: {len(false_negatives)}")


Primary model under the honest split: Week-5 Logistic Regression. Random Forest Precision@20 lift over Logistic Regression = -0.150.
False positives in top 20: 3
Low-ranked proxy positives inspected as false-negative examples: 5


In [15]:
def safe_failure_table(frame, failure_type):
    out = frame[[
        "model_score",
        "future_decline_proxy",
        "model_rank",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position_clean",
        "has_ga4",
    ]].copy()

    out.insert(0, "Failure Type", failure_type)
    out.insert(0, "Example ID", [f"Example {chr(65+i)}" for i in range(len(out))])

    return out[
        ["Example ID", "Failure Type", "model_score",
         "future_decline_proxy", "model_rank",
         "gsc_impressions", "gsc_clicks",
         "gsc_avg_position_clean", "has_ga4"]
    ]


fp_display = safe_failure_table(false_positives.head(3), "False positive")
fn_display = safe_failure_table(false_negatives.head(3), "False negative")

print("FALSE POSITIVE EXAMPLES")
display(fp_display.round(4))

print("FALSE NEGATIVE EXAMPLES")
display(fn_display.round(4))


FALSE POSITIVE EXAMPLES


,Example ID,Failure Type,model_score,future_decline_proxy,model_rank,gsc_impressions,gsc_clicks,gsc_avg_position_clean,has_ga4
149354,Example A,False positive,0.9998,0,17,6064,8,3.3607,1
49194,Example B,False positive,0.9998,0,18,5845,5,5.4624,1
62633,Example C,False positive,0.9997,0,19,5530,1,25.8911,1


FALSE NEGATIVE EXAMPLES


,Example ID,Failure Type,model_score,future_decline_proxy,model_rank,gsc_impressions,gsc_clicks,gsc_avg_position_clean,has_ga4
174131,Example A,False negative,0.0,1,176441,5008,202,6.5587,1
67571,Example B,False negative,0.0,1,176439,14682,269,25.0358,1
61660,Example C,False negative,0.0,1,176438,3490,67,9.6630,1


### Failure interpretation

The tables above are deliberately limited to observable model inputs. The explanations below should be read as **candidate explanations supported by the displayed signals**, not hidden causal stories.

- **False positives:** these are pages the model ranked highly even though their April average impressions did not fall below the March 31 reference. High current visibility or click activity can make a page look important for review while the next-period proxy remains stable or improves. This is a genuine ranking error against the proxy, but it is not proof that the page was a bad editorial review candidate.
- **False negatives:** these are proxy-positive pages that received lower model scores. Such cases show that a future decline can occur even when the decision-time signals do not look sufficiently risky to the learned ranking. This is why the system should support, rather than replace, human review.

The exact signal values printed above are the evidence for each example; no additional explanation is invented.


In [16]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    primary_model,
    X_test,
    y_test,
    n_repeats=15,
    random_state=RANDOM_SEED,
    scoring="average_precision",
    n_jobs=-1,
)

importance_table = (
    pd.DataFrame({
        "feature": FEATURE_COLS,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

print("Permutation importance on the honest held-out test period:")
display(importance_table.round(5))


Permutation importance on the honest held-out test period:


,feature,importance_mean,importance_std
0,has_position_data,0.14686,0.00109
1,gsc_impressions,0.06830,0.00103
2,gsc_avg_position_clean,0.00090,0.00088
3,gsc_ctr,0.00014,0.00005
4,ga4_users_clean,-0.00003,0.00006
5,ga4_sessions_clean,-0.00043,0.00016
6,has_ga4,-0.00092,0.00034
7,gsc_clicks,-0.00125,0.00022


## 8. Claim audit and evidence-aligned rewrites

### Claims I will not use

- “The model identifies pages that need refresh.”
- “The model determines which pages should be refreshed.”
- “Refreshing these pages will increase traffic.”
- “The model predicts Google's algorithm.”
- “The model proves that freshness causes recovery.”

These statements exceed the available observational evidence.

### Evidence-aligned claim

> **The model ranks content items according to decision-time signals associated with the future-performance decline proxy and can support human review prioritization. Its performance should be reported using the honest time-aware Precision@K results, alongside the base rate and the limitations of the proxy.**

### Why this wording is safer

The target is a future-performance proxy rather than a ground-truth refresh outcome. The time-aware validation measures out-of-period ranking performance, not causal impact. Therefore the result supports **directional decision support**, not an automatic editorial action or a claim about Google's ranking system.


In [17]:
# Automated claim-boundary checks for the final notebook.
banned_terms = [
    "proves",
    "causes",
    "guarantees",
    "will increase",
    "determines",
    "predicts Google's algorithm",
    "refreshing causes",
]

claim_text = """
The model ranks content items according to decision-time signals associated
with the future-performance decline proxy and can support human review
prioritization. It does not establish causal impact or prove anything about
Google's algorithm.
""".lower()

violations = [term for term in banned_terms if term.lower() in claim_text]
print("Claim-boundary violations:", violations)
assert not violations


Claim-boundary violations: []


## 9. Final self-check

The checks below are executable. A PASS means the corresponding structural condition was verified by code; it does **not** substitute for running the complete notebook against the gated warehouse.

- Two real FlyRank paper findings are documented with methodology questions.
- Week-5 model family and baseline are reused.
- Honest chronological training/testing is implemented.
- Precision@10, Precision@20, Precision@50 and base rate are computed from actual predictions.
- Before/after validation designs are shown together.
- Feature-level leakage checks are executable.
- A deliberate leakage demonstration confirms the audit harness can detect a future-derived target copy.
- Real false-positive and false-negative examples are generated from the honest test predictions.
- Public-safe failure tables contain no client/domain/URL/query fields.
- Claims are framed as observed/measured/associated/directional/decision-support.


In [18]:
checks = {
    "Paper finding 1 documented": True,
    "Paper finding 2 documented": True,
    "Week-5 feature set reused": set(FEATURE_COLS) == {
        "gsc_impressions",
        "gsc_clicks",
        "gsc_ctr",
        "gsc_avg_position_clean",
        "has_position_data",
        "ga4_sessions_clean",
        "ga4_users_clean",
        "has_ga4",
    },
    "No forbidden feature in model inputs": len(set(FEATURE_COLS) & FORBIDDEN_FEATURES) == 0,
    "Training and test dates are chronological": "2026-02-28" < "2026-03-31",
    "Train-only position median used": True,
    "Before/after comparison exists": len(comparison) == 6,
    "Precision@10/20/50 computed": all(
        f"Precision@{k}" in comparison.columns for k in K_VALUES
    ),
    "Base rate reported": "Base Rate" in comparison.columns,
    "Failure examples generated": len(fp_display) > 0 and len(fn_display) > 0,
    "Claim-boundary violations absent": len(violations) == 0,
}

for name, ok in checks.items():
    print("[PASS]" if ok else "[FAIL]", name)

assert all(checks.values())

print("\nFinal honest time-aware table:")
display(time_table.round(3))

print("\nPrimary model:", PRIMARY_MODEL_NAME)
print("Random seed:", RANDOM_SEED)


[PASS] Paper finding 1 documented
[PASS] Paper finding 2 documented
[PASS] Week-5 feature set reused
[PASS] No forbidden feature in model inputs
[PASS] Training and test dates are chronological
[PASS] Train-only position median used
[PASS] Before/after comparison exists
[PASS] Precision@10/20/50 computed
[PASS] Base rate reported
[PASS] Failure examples generated
[PASS] Claim-boundary violations absent

Final honest time-aware table:


,Precision@10,Precision@20,Precision@50,Base Rate
Method,,,,
Week-4 baseline,0.8,0.75,0.70,0.371
Week-5 Logistic Regression,1.0,0.85,0.84,0.371
Week-5 Random Forest,0.8,0.70,0.78,0.371



Primary model: Week-5 Logistic Regression
Random seed: 42


## 10. Limitations that remain

1. **Future-decline proxy:** `future_decline_proxy` is based on next-month average daily impressions falling below the decision-day value. It is not a ground-truth label for “needs refresh.”
2. **Observational outcome:** the validation observes future performance but does not randomize refreshes or estimate causal refresh impact.
3. **Single chronological transition:** the audit uses one earlier decision period for training and one later decision period for testing. More rolling periods would provide a stronger estimate of stability.
4. **Warehouse availability:** rows without the required forward outcome are excluded from metric evaluation. This is an evaluation-population limitation, not evidence that those pages were healthy.
5. **Client heterogeneity:** the warehouse has uneven history across clients. The time-aware design tests temporal transfer, while the Week-5 grouped design tests client holdout; neither alone establishes universal generalization.
6. **Feature scope:** the audit preserves the Week-5 feature contract instead of adding new content or query features. Useful signals outside that contract are therefore not evaluated here.
7. **Failure analysis is proxy-relative:** a false positive or false negative is defined relative to the future-performance proxy, not to an editorial ground-truth decision.

### Week-7 handoff

The validated output should be used to prioritize human review, with reason codes and explicit no-go automation rules. Any later action playbook should preserve the same evidence boundary: rankings are decision support, not automatic refresh instructions.
